# 3장 타이핑 동행 · 경계신뢰 (핵심 한 줄 채우기)

이 노트북은 경계신뢰 모형의 **핵심 갱신 한 줄**을 여러분이 직접 채워 완성합니다. 나머지 배관(행위자·모형·집계)은 채워져 있습니다. 셀을 위에서부터 Shift+Enter 로 실행하세요.

📖 본문: [3장 §3.2~3.4 읽기](https://grow.minds.kr/textbooks/css-methods/vol0/book/ch03-첫-ABM-경계신뢰.html)


## 1. 준비 (도구 상자 설치)
처음 한 번만, 보통 1분 안팎.


In [ ]:
!pip install -q mesa==3.3.1


## 2. 헬퍼 (그대로 실행)
모형을 정해진 단계만큼 돌려 시계열을 돌려주는 작은 도우미입니다. 읽어만 두고 실행하세요.


In [ ]:
def run(model, steps):
    for _ in range(steps):
        model.step()
    return model.datacollector.get_model_vars_dataframe()


## 3. 모형: 핵심 한 줄을 채우세요 (§3.2)
아래 `Person.step()` 안에 **갱신 규칙 한 줄**만 비어 있습니다. 책 §3.2를 보고 그 한 줄을 채우고, 바로 아래 `raise` 줄을 지운 뒤 이 셀을 실행하세요.

채우는 규칙: 신뢰 범위 안 이웃들의 평균 의견 쪽으로, `mu` 만큼 자기 의견을 옮긴다.


In [ ]:
import numpy as np, mesa

def clusters(model):
    # 소수점 첫째 자리로 반올림한 의견의 종류 수 = 대략의 군집 수
    return len({round(a.opinion, 1) for a in model.agents})

class Person(mesa.Agent):
    def __init__(self, model, opinion):
        super().__init__(model)
        self.opinion = opinion                 # 상태: 의견 (0~1)
    def step(self):
        eps, mu = self.model.eps, self.model.mu
        near = [a.opinion for a in self.model.agents
                if abs(a.opinion - self.opinion) < eps]   # 신뢰 범위 이웃
        if near:
            # ▼▼▼ 여기 한 줄을 채우고 아래 raise 줄을 지우세요 (책 §3.2) ▼▼▼
            # 힌트: 신뢰 범위 이웃 평균 np.mean(near) 쪽으로 mu만큼 이동
            # 꼴:  self.opinion += mu * ( ??? - self.opinion )
            raise NotImplementedError('갱신 규칙 한 줄을 채우세요')
            # ▲▲▲ ------------------------------------------- ▲▲▲

class BoundedConfidence(mesa.Model):
    def __init__(self, n=80, eps=0.2, mu=0.5, seed=None):
        super().__init__(seed=seed)
        self.eps, self.mu = eps, mu
        for _ in range(n):
            Person(self, self.random.random())
        self.datacollector = mesa.DataCollector(model_reporters={'clusters': clusters})
    def step(self):
        self.agents.shuffle_do('step')
        self.datacollector.collect(self)


## 4. 실행하고 서명 확인 (§3.4)
제대로 채웠다면 아래가 돌아가고, 신뢰 범위 `eps` 가 작으면 여러 군집, 크면 합의(1군집)가 나옵니다.

**확인(서명, 씨앗 1)**: `eps 0.05 → 9군집 · 0.15 → 3 · 0.30 → 1`. 이 세 숫자가 나오면 규칙을 맞게 채운 것입니다.


In [ ]:
for eps in (0.05, 0.15, 0.30):
    df = run(BoundedConfidence(eps=eps, seed=1), 40)
    print(f'eps={eps:>4}: 최종 군집 {int(df["clusters"].iloc[-1])}')


## 5. 직접 바꿔 보기
- `eps` 를 0.05~0.40 사이 여러 값으로 바꿔, 군집이 어디서 확 줄어드는지(파편→합의 경계) 찾아보세요.
- 씨앗을 1 대신 73·37 로 바꿔 보세요. 한 번의 실행으로 결론지을 수 있나요? (4장에서 분포로 다룹니다.)

> **검증 로그 (책 부록 B)**: 무엇을 채웠고, eps별 군집이 서명과 맞았는지, 무엇을 바꿔 봤는지 한 문단으로 적어 두세요.
